# Module 7 Lab — Agent Harnesses & Skills

Complete the exercises in order. Docs: open the **7. Agent Harnesses** launcher for the guided walkthrough.

This notebook is the self-contained track: every `# TODO: Exercise …` blank has a collapsible **💡 NEED SOME HELP?** solution right below its cell — try each blank yourself before peeking. (The guided pages follow `harness_lab.py` instead; the two mirror each other, so pick one and stick with it. Full answer key: `harness_lab.answers.ipynb`.)

> Needs `NVIDIA_API_KEY` — set it once with the Secrets Manager; the setup cell below loads it from `secrets.env`.

## Setup — secrets, test data, and the four pi-style core tools

In [ ]:
import argparse
import json
import os
import re
import subprocess
import time
from pathlib import Path

import tiktoken
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# __file__ exists for `python harness_lab.py`; the notebook falls back to its
# own directory (Jupyter kernels start in the notebook's folder).
LAB_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
SKILLS_DIR = LAB_DIR / "skills"
TEST_DATA = LAB_DIR / "test_data" / "sensor_readings.csv"
REPO_ROOT = LAB_DIR.parents[1]

# The Secrets Manager persists keys to <repo>/secrets.env — load them here so
# terminal runs and notebook kernels both see NVIDIA_API_KEY.
load_dotenv(REPO_ROOT / "variables.env")
load_dotenv(REPO_ROOT / "secrets.env")
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "false"  # tracing without a key only 401-spams

MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"


def ensure_test_data():
    """Exercises 3-5 profile/aggregate this CSV; generate it on first use."""
    if not TEST_DATA.exists():
        subprocess.run(["python", str(LAB_DIR / "scripts" / "make_test_data.py")], check=True)

# ---------------------------------------------------------------------------
# The minimal harness, pi-style: a short prompt and four tools.
# ---------------------------------------------------------------------------

MINIMAL_SYSTEM_PROMPT = """You are a capable agent operating a computer through four tools:
read_file, write_file, edit_file, and run_bash.

Environment: run Python as `python` (3.12, pandas/numpy preinstalled) — the
bare `python3` is a different interpreter without those packages.

Work step by step. Use tools to inspect before you act. When writing code,
run it to confirm it works. When the task is complete, reply with a short
summary and no further tool calls."""


@tool
def read_file(path: str) -> str:
    """Read a text file and return its contents (truncated to 8000 chars)."""
    try:
        return Path(path).expanduser().read_text()[:8000]
    except OSError as exc:
        return f"ERROR: {exc}"


@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file, creating parent directories as needed."""
    target = Path(path).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} chars to {target}"


@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Replace the first occurrence of old_text with new_text in a file."""
    target = Path(path).expanduser()
    text = target.read_text()
    if old_text not in text:
        return "ERROR: old_text not found"
    target.write_text(text.replace(old_text, new_text, 1))
    return f"Edited {target}"


@tool
def run_bash(command: str) -> str:
    """Run a shell command and return stdout+stderr (120s timeout)."""
    proc = subprocess.run(
        command, shell=True, capture_output=True, text=True, timeout=120
    )
    return (proc.stdout + proc.stderr)[-8000:] or f"(exit {proc.returncode})"


CORE_TOOLS = [read_file, write_file, edit_file, run_bash]
TOOL_REGISTRY = {t.name: t for t in CORE_TOOLS}


# The harness, not the model, holds the conversation. invoke_with_retry keeps a
# reference to the live message list so Exercise 5 can review the transcript.
LAST_RUN_MESSAGES = []


def invoke_with_retry(model, messages, attempts=3):
    """Harnesses own retries (responsibility #4): survive transient API errors."""
    global LAST_RUN_MESSAGES
    if isinstance(messages, list):
        LAST_RUN_MESSAGES = messages
    for attempt in range(attempts):
        try:
            return model.invoke(messages)
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(2 * (attempt + 1))


def tool_call_count(messages=None) -> int:
    """Tool calls in a recorded run — how hard the harness worked."""
    msgs = LAST_RUN_MESSAGES if messages is None else messages
    return sum(len(getattr(m, "tool_calls", None) or []) for m in msgs)


def skills_consulted(messages=None) -> list:
    """Which skills a recorded run load_skill-ed — the Exercise 4/5 receipts."""
    msgs = LAST_RUN_MESSAGES if messages is None else messages
    return [
        call["args"].get("name", "?")
        for m in msgs
        for call in (getattr(m, "tool_calls", None) or [])
        if call["name"] == "load_skill"
    ]

## Exercise 1 — Build the Minimal Harness

pi proves a complete harness needs surprisingly little: a short system prompt, four tools, and a loop. Build exactly that around Nemotron.

In [ ]:
def build_bare_agent(extra_tools=None, system_prompt=MINIMAL_SYSTEM_PROMPT):
    """Exercise 1: a complete harness in ~20 lines.

    Returns run(task) -> final answer string. The loop: call the model,
    execute any tool calls, feed results back, repeat until a plain reply.
    """
    tools = CORE_TOOLS + list(extra_tools or [])
    registry = {t.name: t for t in tools}

    # TODO: Exercise 1a — create the model and bind the tools to it.
    # Use ChatNVIDIA with MODEL_NAME, temperature=0.2,
    # max_completion_tokens=4096 (the 1024 default truncates long write_file
    # calls mid-JSON), and timeout=180 (a 120B model can exceed the 60s
    # default on long generations), then .bind_tools(tools)
    model = None

    def run(task: str, max_turns: int = 20) -> str:
        messages = [SystemMessage(content=system_prompt), HumanMessage(content=task)]
        for _ in range(max_turns):
            # TODO: Exercise 1b — implement the agentic loop:
            #   1. call invoke_with_retry(model, messages) and append the response
            #   2. if the response has no .tool_calls, return response.content
            #   3. otherwise, for each tool call: print a one-line trace
            #      (f"  🛠️  {call['name']}({json.dumps(call['args'])[:120]})") so you
            #      can watch the harness work, then execute it via `registry` —
            #      catching any exception as an f"ERROR: ..." result so the model
            #      can correct itself — and append a ToolMessage(content=str(result),
            #      tool_call_id=call["id"])
            raise NotImplementedError("Complete Exercise 1b")
        return "ERROR: max turns exceeded"

    return run

<details>
<summary>💡 NEED SOME HELP? — Exercise 1a (create + bind the model)</summary>
    
---

```python
model = ChatNVIDIA(
    model=MODEL_NAME, temperature=0.2, max_completion_tokens=4096, timeout=180
).bind_tools(tools)
```

---

Skip `.bind_tools(tools)` and the model never sees the tool schemas — it will answer in prose instead of requesting `write_file`, and the loop will exit on the first turn having done nothing.
</details>

<details>
<summary>💡 NEED SOME HELP? — Exercise 1b (the agentic loop)</summary>
    
---

```python
response = invoke_with_retry(model, messages)
messages.append(response)
if not response.tool_calls:
    return response.content
for call in response.tool_calls:
    print(f"  🛠️  {call['name']}({json.dumps(call['args'])[:120]})")
    try:
        result = registry[call["name"]].invoke(call["args"])
    except Exception as exc:
        result = f"ERROR: {type(exc).__name__}: {str(exc)[:500]}"
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
```

---

Four details that matter: the `print` is what makes the loop observable — without it the harness runs silently and you only see the final answer; use the loop's local `registry` (not the module-level `TOOL_REGISTRY`) so extra tools like `load_skill` stay callable in later exercises; execute tools with `.invoke(call["args"])` — LangChain tool objects are not plain functions you can call directly; and feed tool errors back as the `ToolMessage` instead of letting them crash the loop — reading its own error is what lets the agent self-correct.
</details>

In [ ]:
run = build_bare_agent()
print(run(
    "Create a file named harness_hello.txt containing the words "
    "'minimal harness', then read it back and confirm its contents."
))

## Exercise 2 — Measure the Context Tax

Count what your harness pays per turn — prompt plus tool schemas — against a maximal configuration, then implement lazy skill loading.

In [ ]:
ENCODER = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))


def harness_overhead(system_prompt: str, tools) -> int:
    """Tokens a harness pays on EVERY call: prompt + registered tool schemas."""
    # TODO: Exercise 2a — return the token count of the system prompt PLUS
    # the JSON-serialized tool schemas. Run every tool through
    # convert_to_openai_tool() — it accepts both tool objects and
    # already-converted dict schemas.
    raise NotImplementedError("Complete Exercise 2a")


def measure_context_tax() -> dict:
    """Exercise 2a: compare minimal vs maximal per-turn overhead."""
    maximal_prompt = (LAB_DIR / "maximal_system_prompt.txt").read_text()
    maximal_tools = json.loads((LAB_DIR / "maximal_tool_schemas.json").read_text())

    minimal = harness_overhead(MINIMAL_SYSTEM_PROMPT, CORE_TOOLS)
    maximal = harness_overhead(maximal_prompt, maximal_tools)

    print(f"Minimal harness: {minimal:>7,} tokens/turn")
    print(f"Maximal harness: {maximal:>7,} tokens/turn   ({maximal / minimal:.1f}x tax)")
    return {"minimal": minimal, "maximal": maximal}


def parse_frontmatter(skill_md: str) -> dict:
    """Pull name/description out of a SKILL.md YAML frontmatter block."""
    match = re.match(r"^---\n(.*?)\n---\n", skill_md, re.DOTALL)
    if not match:
        raise ValueError("SKILL.md missing frontmatter")
    meta = {}
    for line in match.group(1).splitlines():
        if ":" in line:
            key, _, value = line.partition(":")
            meta[key.strip()] = value.strip()
    if not meta.get("name") or not meta.get("description"):
        raise ValueError("frontmatter needs both name and description")
    return meta


def load_skills_lazily(skills_dir: Path = SKILLS_DIR):
    """Exercise 2b: lazy skills — one line of context each, full body on demand.

    Returns (index_text, load_skill_tool). The index goes in the system
    prompt; the tool lets the model pull in a full skill body when needed.
    """
    bodies, index_lines = {}, []
    for skill_file in sorted(skills_dir.glob("*/SKILL.md")):
        if skill_file.parent.name.startswith("."):
            continue
        # TODO: Exercise 2b(i) — read the file, parse_frontmatter() it, store
        # the full text in bodies[name], and append "- {name}: {description}"
        # to index_lines.
        raise NotImplementedError("Complete Exercise 2b(i)")

    index_text = (
        "Installed skills — before starting a task, load any skill that covers "
        "it with the load_skill tool and follow its instructions. When in "
        "doubt, load it:\n"
        + "\n".join(index_lines)
    )

    @tool
    def load_skill(name: str) -> str:
        """Load the full instructions of an installed skill by name."""
        # TODO: Exercise 2b(ii) — return the stored full body for `name`,
        # or an error string if no such skill exists.
        raise NotImplementedError("Complete Exercise 2b(ii)")

    eager = sum(count_tokens(b) for b in bodies.values())
    lazy = count_tokens(index_text)
    print(f"{len(bodies)} eager skills: +{eager:,} tokens/turn")
    print(f"{len(bodies)} lazy skills:  +{lazy:,} tokens/turn   ({eager / max(lazy, 1):.0f}x savings)")
    return index_text, load_skill

<details>
<summary>💡 NEED SOME HELP? — Exercise 2a (harness_overhead)</summary>

---

```python
schemas = [convert_to_openai_tool(t) for t in tools]
return count_tokens(system_prompt) + count_tokens(json.dumps(schemas))
```

---

Counting only the prompt misses the point — the schemas are the bigger half of a maximal harness's tax. Don't `callable()`-gate the conversion: the maximal tools arrive from JSON as plain dicts, and `convert_to_openai_tool()` passes those through untouched.
</details>

<details>
<summary>💡 NEED SOME HELP? — Exercise 2b(i) (build the one-line index)</summary>

---

```python
text = skill_file.read_text()
meta = parse_frontmatter(text)
bodies[meta["name"]] = text
index_lines.append(f"- {meta['name']}: {meta['description']}")
```

---

If you put the full body in the index you've rebuilt eager loading — the very bug this exercise exposes. The index gets one line per skill; the bodies stay in the dict until asked for.
</details>

<details>
<summary>💡 NEED SOME HELP? — Exercise 2b(ii) (load_skill)</summary>

---

```python
return bodies.get(name, f"ERROR: no skill named {name!r}")
```

---

Return an error *string* on a miss (don't raise) — the model should read the error and recover.
</details>

In [ ]:
measure_context_tax()
_ = load_skills_lazily()

## Exercise 3 — Author a Portable Skill

Write `skills/dataset_profiler/SKILL.md` (format: `skills/code_review/SKILL.md` at the repo root), then run it through the lazy loader. The same folder also drops unchanged into Hermes's `~/.hermes/skills/` — one skill, two harnesses.

<details>
<summary>💡 NEED SOME HELP? — Exercise 3 (the SKILL.md shape)</summary>

The `description` line is what triggers skill loading — make it match the task vocabulary ("profile, summarize, or explore an unfamiliar CSV or DataFrame"), not the implementation. A good body gives the agent a numbered procedure and an output format:

```markdown
---
name: dataset_profiler
description: Systematic procedure for profiling, summarizing, or exploring an unfamiliar CSV file or DataFrame
---

# Dataset Profiler Skill

Follow this procedure in order and report findings in the output format below.

1. **Shape & size** — row count, column count, file size on disk.
2. **Schema** — every column with its dtype; flag dtypes that look wrong.
3. **Nulls** — per-column null counts; call out any column over 5% null.
4. **Duplicates** — count of fully duplicated rows.
5. **Numeric distributions** — min / max / mean / std; flag impossible values.
6. **Cardinality** — unique counts; likely categoricals vs identifiers.
7. **Three surprising facts** — the most decision-relevant findings.

## Output format
(a compact template the agent fills in)
```

A fully worked version lives at `skills/.examples/dataset_profiler/SKILL.md` — but draft yours first; a skill you authored yourself is the one worth carrying across harnesses.
</details>

In [ ]:
def run_with_skills(task: str) -> str:
    """Run the bare agent with the lazy skill index attached."""
    index_text, load_skill = load_skills_lazily()
    run = build_bare_agent(
        extra_tools=[load_skill],
        system_prompt=MINIMAL_SYSTEM_PROMPT + "\n\n" + index_text,
    )
    return run(task)

In [ ]:
ensure_test_data()
print(run_with_skills(
    f"Profile the dataset at {TEST_DATA} and report your findings."
))

## Exercise 4 — Verified NVIDIA Skill, Real GPU

Install and signature-verify the NVIDIA `accelerated-computing-cudf` skill:

```bash
bash scripts/install_nvidia_skill.sh accelerated-computing-cudf
```

Then open a terminal, keep `watch -n 0.5 nvidia-smi` visible, and run the next cell — you'll see your GPU light up while the agent works.

In [ ]:
def run_gpu_task() -> str:
    """Aggregate a 1M-row CSV ×10; the cuDF skill — not the task — steers the
    model to the GPU.

    Install + verify the skill first:
      bash scripts/install_nvidia_skill.sh accelerated-computing-cudf
    """
    if not (SKILLS_DIR / "accelerated-computing-cudf" / "SKILL.md").exists():
        return "Skill not installed — run scripts/install_nvidia_skill.sh first."
    ensure_test_data()

    has_gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
    has_cudf = subprocess.run(["python", "-c", "import cudf"], capture_output=True).returncode == 0
    if not has_gpu:
        print("⚠️  No GPU detected — the agent will fall back to pandas. "
              "On a GPU machine, watch `nvidia-smi` light up instead.")
    elif not has_cudf:
        print("⚠️  cuDF isn't importable — run `pip install cudf-cu12`, "
              "or the agent will fall back to pandas.")

    # The task asks for speed but never names the GPU — the skill supplies
    # the how; the receipt below catches the model skipping it.
    result = run_with_skills(
        f"Load {TEST_DATA} (about 1M rows) and compute the mean, max, and count "
        "of `reading` per `device_id`, sorted by mean descending. Repeat the "
        "full load-and-aggregate 10 times in a loop — don't hoist the CSV read "
        "out of the loop — and make it fast: use the best-performing DataFrame "
        "stack available on this machine. Save the final result to "
        f"{LAB_DIR / 'test_data' / 'aggregates.csv'} and show the top 5 rows."
    )
    if "accelerated-computing-cudf" in skills_consulted():
        return f"{result}\n\n🧾 Receipt: the agent loaded the verified skill before computing."
    return (f"{result}\n\n🧾 Receipt: the agent never loaded the verified skill "
            "this run — rerun and watch for load_skill.")

In [ ]:
print(run_gpu_task())

<details>
<summary>💡 NEED SOME HELP? — Exercise 4 (GPU stays at zero?)</summary>

- **"Skill not installed"** — run `bash scripts/install_nvidia_skill.sh accelerated-computing-cudf` from `code/7-agent-harnesses/` first.
- **GPU utilization stays at 0** — check the dataset actually crossed the 100K-row size gate the skill teaches (the generator script makes 1M rows by default), and confirm cuDF imports GPU-side: `python -c "import cudf; print(cudf.__version__)"`. If that fails: `pip install cudf-cu12`.
- **The 🧾 receipt says the skill was never consulted** — rerun the cell; the load decision is the model's.
- **No GPU on this machine** — the cell warns you up front and the agent falls back to pandas; the same aggregation still completes, just CPU-slow.
</details>

## Exercise 5 — The Self-Evolving Harness

The pi finale: after finishing a task, the agent reviews its own transcript, writes a new SKILL.md, and uses it on the next run.

In [ ]:
SKILL_AUTHOR_PROMPT = """Review this transcript of an agent completing a task.
Extract the reusable PROCEDURE (not the task-specific values) and write it as
an agent skill in exactly this format — output ONLY the file content:

---
name: <short_snake_case_name>
description: <one line stating when this skill should be used>
---

# <Title>

<numbered procedure the agent should follow next time>

TRANSCRIPT:
{transcript}"""


def format_transcript(messages) -> str:
    """Flatten a run's message list into the TASK/TOOL/RESULT/ANSWER transcript."""
    lines = []
    for msg in messages:
        if isinstance(msg, SystemMessage):
            continue
        if isinstance(msg, HumanMessage):
            lines.append(f"TASK: {msg.content}")
        elif isinstance(msg, ToolMessage):
            lines.append(f"RESULT: {str(msg.content)[:300]}")
        elif getattr(msg, "tool_calls", None):
            lines.extend(
                f"TOOL: {call['name']}({json.dumps(call['args'])[:300]})"
                for call in msg.tool_calls
            )
        elif getattr(msg, "content", None):
            lines.append(f"ANSWER: {msg.content}")
    return "\n".join(lines)


def self_evolve_skill(transcript: str, skills_dir: Path = SKILLS_DIR) -> Path:
    """Exercise 5: the agent writes a new skill from its own transcript."""
    model = ChatNVIDIA(
        model=MODEL_NAME, temperature=0.2, max_completion_tokens=4096, timeout=180
    )

    # TODO: Exercise 5 — make the agent author its own skill:
    #   1. invoke_with_retry(model, SKILL_AUTHOR_PROMPT.format(transcript=...))
    #   2. strip any ``` fences from the response content
    #   3. parse_frontmatter() to VALIDATE before saving — a malformed skill
    #      breaks the lazy loader on the next run (Module 6 lesson!)
    #   4. save to skills_dir / meta["name"] / "SKILL.md" and return the path
    raise NotImplementedError("Complete Exercise 5")


def run_self_evolution_demo():
    task = (
        f"Check whether the CSV at {TEST_DATA} has any nulls or duplicate "
        "rows, and report the verdict in one sentence."
    )
    ensure_test_data()

    run = build_bare_agent()

    print("=== Run 1 (no skill) ===")
    print(run(task))
    run1_calls = tool_call_count()
    # The agent reviews the REAL transcript of run 1 — every tool call and
    # result the harness recorded — and distills the reusable procedure.
    self_evolve_skill(format_transcript(LAST_RUN_MESSAGES))

    print("\n=== Run 2 (with the skill the agent just wrote) ===")
    print(run_with_skills(task))
    run2_calls = tool_call_count()
    loaded = skills_consulted()
    print(f"\n🧾 Run 1: {run1_calls} tool calls · Run 2: {run2_calls} tool calls "
          + (f"(consulted: {', '.join(loaded)})" if loaded else "(no skill loaded this run)"))

<details>
<summary>💡 NEED SOME HELP? — Exercise 5 (self_evolve_skill)</summary>

---

````python
prompt = SKILL_AUTHOR_PROMPT.format(transcript=transcript)
skill_md = invoke_with_retry(model, prompt).content
skill_md = skill_md.strip().removeprefix("```markdown").removeprefix("```").removesuffix("```").strip()

meta = parse_frontmatter(skill_md)  # validate BEFORE saving
target = skills_dir / meta["name"] / "SKILL.md"
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(skill_md)
print(f"🌱 Agent wrote itself a new skill: {target}")
return target
````

---

The `parse_frontmatter()` call is the line to internalize: validate *before* anything lands in the skills directory, not after the loader crashes on the next run — exactly the self-evolution failure Module 6 warned about.
</details>

In [ ]:
run_self_evolution_demo()